
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab - Creating Bronze Tables from JSON Files
### Duration: ~ 15 minutes

In this lab you will ingest a JSON file as Delta table and then flatten the JSON formatted string column.

### Learning Objectives
  - Inspect a raw JSON file.
  - Read in JSON files to a Delta table and flatten the JSON formatted string column.

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default and you have a Shared SQL warehouse.

<!-- ![Select Cluster](./Includes/images/selecting_cluster_info.png) -->

Follow these steps to select the classic compute cluster:


1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.

   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.


## A. Classroom Setup

Run the following cell to configure your working environment for this notebook.

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course in the lab environment.

In [0]:
%run ./Includes/Classroom-Setup-07L

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


----------------------------------------------------------------------------------------
Directory /Volumes/dbacademy/ops/labuser14289851_1774380005@vocareum_com/csv_demo_files already exists. No action taken.
Directory /Volumes/dbacademy/ops/labuser14289851_1774380005@vocareum_com/json_demo_files already exists. No action taken.
Directory /Volumes/dbacademy/ops/labuser14289851_1774380005@vocareum_com/xml_demo_files already exists. No action taken.
----------------------------------------------------------------------------------------



Generated file: /Volumes/dbacademy/ops/labuser14289851_1774380005@vocareum_com/json_demo_files/lab_kafka_events.json


File written with 1 malformed 'timestamp' at position 0: /Volumes/dbacademy/ops/labuser14289851_1774380005@vocareum_com/json_demo_files/lab_kafka_events_challenge.json


Run the cell below to view your default catalog and schema. Notice that your default catalog is **dbacademy** and your default schema is your unique **labuser** schema.

**NOTE:** The default catalog and schema are pre-configured for you to avoid the need to specify the three-level name when writing your tables (i.e., catalog.schema.table).

In [0]:
SELECT current_catalog(), current_schema()

current_catalog(),current_schema()
dbacademy,labuser14289851_1774380005


## B. Lab - JSON Ingestion
**Scenario:** You are working with your data team on ingesting a JSON file into Databricks. Your job is to ingest the JSON file as is into a bronze table, then create a second bronze table that flattens the JSON formatted string column in the raw bronze table for downstream processing.

### B1. Inspect the Dataset

1. In the cell below, view the value of the Python variable `DA.paths.working_dir`. This variable will reference your **dbacademy.ops.labuser** volume, as each user has a different source volume. This variable is created within the classroom setup script to dynamically reference your unique volume.

   Run the cell and review the results. You’ll notice that the `DA.paths.working_dir` variable points to your `/Volumes/dbacademy/ops/labuser` volume.

**Note:** Instead of using the `DA.paths.working_dir` variable, you could also specify the path name directly by right clicking on your volume and selecting **Copy volume path**. 

In [0]:
%python
print(DA.paths.working_dir)

/Volumes/dbacademy/ops/labuser14289851_1774380005@vocareum_com


Run the cell to view the data in the `/Volumes/dbacademy/ops/your-labuser-name/json_demo_files/lab_kafka_events.json` file in the location from above.

In [0]:
%python
spark.sql(f'''
          SELECT * 
          FROM json.`{DA.paths.working_dir}/json_demo_files/lab_kafka_events.json`
          ''').display()

key,timestamp,value
event_0,1.774387532258797E9,eyJ1c2VyX2lkIjogInVzZXJfMTgyNCIsICJldmVudF90eXBlIjogImNsaWNrIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTAzLTI0VDIxOjI1OjMyLjI1ODczMiIsICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiaXRlbV85MzYiLCAicXVhbnRpdHkiOiAxLCAicHJpY2VfdXNkIjogNjQuODF9LCB7Iml0ZW1faWQiOiAiaXRlbV8yNDMiLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogNjYuNjZ9LCB7Iml0ZW1faWQiOiAiaXRlbV84MDEiLCAicXVhbnRpdHkiOiA0LCAicHJpY2VfdXNkIjogNjguNzh9XX0=
event_1,1.774387532258836E9,eyJ1c2VyX2lkIjogInVzZXJfNjM5OCIsICJldmVudF90eXBlIjogInZpZXciLCAiZXZlbnRfdGltZXN0YW1wIjogIjIwMjYtMDMtMjRUMjE6MjU6MzIuMjU4ODA0IiwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJpdGVtXzgyNCIsICJxdWFudGl0eSI6IDIsICJwcmljZV91c2QiOiA0NC4wNH0sIHsiaXRlbV9pZCI6ICJpdGVtXzc1OCIsICJxdWFudGl0eSI6IDIsICJwcmljZV91c2QiOiA3OC42N30sIHsiaXRlbV9pZCI6ICJpdGVtXzU3MyIsICJxdWFudGl0eSI6IDEsICJwcmljZV91c2QiOiAzMi42Mn1dfQ==
event_2,1.774387532258861E9,eyJ1c2VyX2lkIjogInVzZXJfNDUzNyIsICJldmVudF90eXBlIjogImNsaWNrIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTAzLTI0VDIxOjI1OjMyLjI1ODg0MCIsICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiaXRlbV85MzQiLCAicXVhbnRpdHkiOiA0LCAicHJpY2VfdXNkIjogOTkuMjh9LCB7Iml0ZW1faWQiOiAiaXRlbV84MDYiLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogOTAuNDJ9XX0=
event_3,1.774387532258889E9,eyJ1c2VyX2lkIjogInVzZXJfNzk5OSIsICJldmVudF90eXBlIjogInZpZXciLCAiZXZlbnRfdGltZXN0YW1wIjogIjIwMjYtMDMtMjRUMjE6MjU6MzIuMjU4ODY1IiwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJpdGVtXzc3OSIsICJxdWFudGl0eSI6IDEsICJwcmljZV91c2QiOiAyNC42Mn0sIHsiaXRlbV9pZCI6ICJpdGVtXzU1NyIsICJxdWFudGl0eSI6IDQsICJwcmljZV91c2QiOiA3Ni40MX0sIHsiaXRlbV9pZCI6ICJpdGVtXzIwMiIsICJxdWFudGl0eSI6IDMsICJwcmljZV91c2QiOiA4My43OH1dfQ==
event_4,1.774387532258913E9,eyJ1c2VyX2lkIjogInVzZXJfMTgwOCIsICJldmVudF90eXBlIjogInZpZXciLCAiZXZlbnRfdGltZXN0YW1wIjogIjIwMjYtMDMtMjRUMjE6MjU6MzIuMjU4ODkzIiwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJpdGVtXzYyNyIsICJxdWFudGl0eSI6IDQsICJwcmljZV91c2QiOiA3NC4xN30sIHsiaXRlbV9pZCI6ICJpdGVtXzQ2NyIsICJxdWFudGl0eSI6IDQsICJwcmljZV91c2QiOiAxNi45Mn1dfQ==
event_5,1.774387532258931E9,eyJ1c2VyX2lkIjogInVzZXJfODAxMiIsICJldmVudF90eXBlIjogImNsaWNrIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTAzLTI0VDIxOjI1OjMyLjI1ODkxNiIsICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiaXRlbV8xMzkiLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogOTkuNDN9XX0=
event_6,1.774387532258961E9,eyJ1c2VyX2lkIjogInVzZXJfMTg3OCIsICJldmVudF90eXBlIjogInB1cmNoYXNlIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTAzLTI0VDIxOjI1OjMyLjI1ODkzNSIsICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiaXRlbV81MzAiLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogOTUuMDJ9LCB7Iml0ZW1faWQiOiAiaXRlbV8xODUiLCAicXVhbnRpdHkiOiAxLCAicHJpY2VfdXNkIjogOTQuODl9LCB7Iml0ZW1faWQiOiAiaXRlbV82NzUiLCAicXVhbnRpdHkiOiAyLCAicHJpY2VfdXNkIjogNDQuNH1dfQ==
event_7,1.774387532258983E9,eyJ1c2VyX2lkIjogInVzZXJfMzE5OCIsICJldmVudF90eXBlIjogInZpZXciLCAiZXZlbnRfdGltZXN0YW1wIjogIjIwMjYtMDMtMjRUMjE6MjU6MzIuMjU4OTY0IiwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJpdGVtXzIxMiIsICJxdWFudGl0eSI6IDEsICJwcmljZV91c2QiOiA3MS4wNH0sIHsiaXRlbV9pZCI6ICJpdGVtXzEyMSIsICJxdWFudGl0eSI6IDQsICJwcmljZV91c2QiOiAxMS4yNn1dfQ==
event_8,1.774387532259001E9,eyJ1c2VyX2lkIjogInVzZXJfMTA2OSIsICJldmVudF90eXBlIjogImNsaWNrIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTAzLTI0VDIxOjI1OjMyLjI1ODk4NiIsICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiaXRlbV80NzciLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogNTguMDR9XX0=
event_9,1.774387532259023E9,eyJ1c2VyX2lkIjogInVzZXJfODg4NCIsICJldmVudF90eXBlIjogInB1cmNoYXNlIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTAzLTI0VDIxOjI1OjMyLjI1OTAwNCIsICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiaXRlbV80MDMiLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogODguMjN9LCB7Iml0ZW1faWQiOiAiaXRlbV8xNDYiLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogODIuODN9XX0=


### B2. Create the Raw Bronze Table

Inspect and run the code below to ingest the raw JSON file `/Volumes/dbacademy/ops/your-labuser-name/json_demo_files/lab_kafka_events.json` and create the **lab7_lab_kafka_events_raw** table.

Notice the following:
- The **value** column is decoded.
- The **decoded_value** column was created and returns the decoded column as a JSON-formatted string.


In [0]:
CREATE OR REPLACE TABLE lab7_lab_kafka_events_raw
AS
SELECT 
  *,
  cast(unbase64(value) as STRING) as decoded_value
FROM read_files(
        DA.paths_working_dir || '/json_demo_files/lab_kafka_events.json',
        format => "json", 
        schema => '''
          key STRING, 
          timestamp DOUBLE, 
          value STRING
        ''',
        rescueddatacolumn => '_rescued_data'
      );

-- View the table
SELECT *
FROM lab7_lab_kafka_events_raw;

key,timestamp,value,_rescued_data,decoded_value
event_0,1.774387532258797E9,eyJ1c2VyX2lkIjogInVzZXJfMTgyNCIsICJldmVudF90eXBlIjogImNsaWNrIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTAzLTI0VDIxOjI1OjMyLjI1ODczMiIsICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiaXRlbV85MzYiLCAicXVhbnRpdHkiOiAxLCAicHJpY2VfdXNkIjogNjQuODF9LCB7Iml0ZW1faWQiOiAiaXRlbV8yNDMiLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogNjYuNjZ9LCB7Iml0ZW1faWQiOiAiaXRlbV84MDEiLCAicXVhbnRpdHkiOiA0LCAicHJpY2VfdXNkIjogNjguNzh9XX0=,null,"{""user_id"": ""user_1824"", ""event_type"": ""click"", ""event_timestamp"": ""2026-03-24T21:25:32.258732"", ""items"": [{""item_id"": ""item_936"", ""quantity"": 1, ""price_usd"": 64.81}, {""item_id"": ""item_243"", ""quantity"": 3, ""price_usd"": 66.66}, {""item_id"": ""item_801"", ""quantity"": 4, ""price_usd"": 68.78}]}"
event_1,1.774387532258836E9,eyJ1c2VyX2lkIjogInVzZXJfNjM5OCIsICJldmVudF90eXBlIjogInZpZXciLCAiZXZlbnRfdGltZXN0YW1wIjogIjIwMjYtMDMtMjRUMjE6MjU6MzIuMjU4ODA0IiwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJpdGVtXzgyNCIsICJxdWFudGl0eSI6IDIsICJwcmljZV91c2QiOiA0NC4wNH0sIHsiaXRlbV9pZCI6ICJpdGVtXzc1OCIsICJxdWFudGl0eSI6IDIsICJwcmljZV91c2QiOiA3OC42N30sIHsiaXRlbV9pZCI6ICJpdGVtXzU3MyIsICJxdWFudGl0eSI6IDEsICJwcmljZV91c2QiOiAzMi42Mn1dfQ==,null,"{""user_id"": ""user_6398"", ""event_type"": ""view"", ""event_timestamp"": ""2026-03-24T21:25:32.258804"", ""items"": [{""item_id"": ""item_824"", ""quantity"": 2, ""price_usd"": 44.04}, {""item_id"": ""item_758"", ""quantity"": 2, ""price_usd"": 78.67}, {""item_id"": ""item_573"", ""quantity"": 1, ""price_usd"": 32.62}]}"
event_2,1.774387532258861E9,eyJ1c2VyX2lkIjogInVzZXJfNDUzNyIsICJldmVudF90eXBlIjogImNsaWNrIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTAzLTI0VDIxOjI1OjMyLjI1ODg0MCIsICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiaXRlbV85MzQiLCAicXVhbnRpdHkiOiA0LCAicHJpY2VfdXNkIjogOTkuMjh9LCB7Iml0ZW1faWQiOiAiaXRlbV84MDYiLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogOTAuNDJ9XX0=,null,"{""user_id"": ""user_4537"", ""event_type"": ""click"", ""event_timestamp"": ""2026-03-24T21:25:32.258840"", ""items"": [{""item_id"": ""item_934"", ""quantity"": 4, ""price_usd"": 99.28}, {""item_id"": ""item_806"", ""quantity"": 3, ""price_usd"": 90.42}]}"
event_3,1.774387532258889E9,eyJ1c2VyX2lkIjogInVzZXJfNzk5OSIsICJldmVudF90eXBlIjogInZpZXciLCAiZXZlbnRfdGltZXN0YW1wIjogIjIwMjYtMDMtMjRUMjE6MjU6MzIuMjU4ODY1IiwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJpdGVtXzc3OSIsICJxdWFudGl0eSI6IDEsICJwcmljZV91c2QiOiAyNC42Mn0sIHsiaXRlbV9pZCI6ICJpdGVtXzU1NyIsICJxdWFudGl0eSI6IDQsICJwcmljZV91c2QiOiA3Ni40MX0sIHsiaXRlbV9pZCI6ICJpdGVtXzIwMiIsICJxdWFudGl0eSI6IDMsICJwcmljZV91c2QiOiA4My43OH1dfQ==,null,"{""user_id"": ""user_7999"", ""event_type"": ""view"", ""event_timestamp"": ""2026-03-24T21:25:32.258865"", ""items"": [{""item_id"": ""item_779"", ""quantity"": 1, ""price_usd"": 24.62}, {""item_id"": ""item_557"", ""quantity"": 4, ""price_usd"": 76.41}, {""item_id"": ""item_202"", ""quantity"": 3, ""price_usd"": 83.78}]}"
event_4,1.774387532258913E9,eyJ1c2VyX2lkIjogInVzZXJfMTgwOCIsICJldmVudF90eXBlIjogInZpZXciLCAiZXZlbnRfdGltZXN0YW1wIjogIjIwMjYtMDMtMjRUMjE6MjU6MzIuMjU4ODkzIiwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJpdGVtXzYyNyIsICJxdWFudGl0eSI6IDQsICJwcmljZV91c2QiOiA3NC4xN30sIHsiaXRlbV9pZCI6ICJpdGVtXzQ2NyIsICJxdWFudGl0eSI6IDQsICJwcmljZV91c2QiOiAxNi45Mn1dfQ==,null,"{""user_id"": ""user_1808"", ""event_type"": ""view"", ""event_timestamp"": ""2026-03-24T21:25:32.258893"", ""items"": [{""item_id"": ""item_627"", ""quantity"": 4, ""price_usd"": 74.17}, {""item_id"": ""item_467"", ""quantity"": 4, ""price_usd"": 16.92}]}"
event_5,1.774387532258931E9,eyJ1c2VyX2lkIjogInVzZXJfODAxMiIsICJldmVudF90eXBlIjogImNsaWNrIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTAzLTI0VDIxOjI1OjMyLjI1ODkxNiIsICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiaXRlbV8xMzkiLCAicXVhbnRpdHkiOiAzLCAicHJpY2VfdXNkIjogOTkuNDN9XX0=,null,"{""user_id"": ""user_8012"", ""event_type"": ""click"", ""event_timestamp"": ""2026-03-24T21:25:32.258916"", ""items"": [{""item_id"": ""item_139"", ""quantity"": 3, ""price_usd"": 99.43}]}"
event_6,1.774387532258961E9,eyJ1c2VyX2lkIjogInVzZXJfMTg3OCIsICJldmVudF9

### B3. Create the Flattened Bronze Table

1. Your goal is to flatten the JSON formatted string column **decoded_value** from the table **lab7_lab_kafka_events_raw** to create a new table named **lab7_lab_kafka_events_flattened** for downstream processing. The table should contain the following columns:
    - **key**
    - **timestamp**
    - **user_id**
    - **event_type**
    - **event_timestamp**
    - **items**

    You can use whichever technique you prefer:

    - Parse the JSON formatted string (easiest) to flatten
      - [Query JSON strings](https://docs.databricks.com/aws/en/semi-structured/json):

    - Convert the JSON formatted string as a VARIANT and flatten
      - [parse_json function](https://docs.databricks.com/gcp/en/sql/language-manual/functions/parse_json)

    - Convert the JSON formatted string to a STRUCT and flatten
      - [schema_of_json function](https://docs.databricks.com/aws/en/sql/language-manual/functions/schema_of_json)
      - [from_json function](https://docs.databricks.com/gcp/en/sql/language-manual/functions/from_json)

**NOTE:** View the lab solution notebook to view the solutions for each.

2. To begin, run the code below to view the final solution table **lab7_lab_kafka_events_flattened_solution**. This will give you an idea of what your final table should look like.

  **NOTE**: Depending on your solution, the data types of the columns may vary slightly.  


##### Optional Challenge

  As a challenge, after flattening the table, try converting the data types accordingly. Depending on your skill set, you may not convert all columns to the correct data types within the allotted time.

  - **key** STRING
  - **timestamp** DOUBLE
  - **user_id** STRING
  - **event_type** STRING
  - **event_timestamp** TIMESTAMP
  - **items** (STRUCT or VARIANT) depending on the method you used.

In [0]:
SELECT *
FROM lab7_lab_kafka_events_flattened_solution

key,timestamp,user_id,event_type,event_timestamp,items
event_0,1.774387532258797E9,user_1824,click,2026-03-24T21:25:32.258732Z,"List(List(item_936, 64.81, 1), List(item_243, 66.66, 3), List(item_801, 68.78, 4))"
event_1,1.774387532258836E9,user_6398,view,2026-03-24T21:25:32.258804Z,"List(List(item_824, 44.04, 2), List(item_758, 78.67, 2), List(item_573, 32.62, 1))"
event_2,1.774387532258861E9,user_4537,click,2026-03-24T21:25:32.25884Z,"List(List(item_934, 99.28, 4), List(item_806, 90.42, 3))"
event_3,1.774387532258889E9,user_7999,view,2026-03-24T21:25:32.258865Z,"List(List(item_779, 24.62, 1), List(item_557, 76.41, 4), List(item_202, 83.78, 3))"
event_4,1.774387532258913E9,user_1808,view,2026-03-24T21:25:32.258893Z,"List(List(item_627, 74.17, 4), List(item_467, 16.92, 4))"
event_5,1.774387532258931E9,user_8012,click,2026-03-24T21:25:32.258916Z,"List(List(item_139, 99.43, 3))"
event_6,1.774387532258961E9,user_1878,purchase,2026-03-24T21:25:32.258935Z,"List(List(item_530, 95.02, 3), List(item_185, 94.89, 1), List(item_675, 44.4, 2))"
event_7,1.774387532258983E9,user_3198,view,2026-03-24T21:25:32.258964Z,"List(List(item_212, 71.04, 1), List(item_121, 11.26, 4))"
event_8,1.774387532259001E9,user_1069,click,2026-03-24T21:25:32.258986Z,"List(List(item_477, 58.04, 3))"
event_9,1.774387532259023E9,user_8884,purchase,2026-03-24T21:25:32.259004Z,"List(List(item_403, 88.23, 3), List(item_146, 82.83, 3))"


3. Write the query in the cell below to read the **lab_kafka_events_raw** table and create the flattened table **lab7_lab_kafka_events_flattened** following the requirements from above.

In [0]:
DESCRIBE lab7_lab_kafka_events_raw;

col_name,data_type,comment
key,string,null
timestamp,double,null
value,string,null
_rescued_data,string,null
decoded_value,string,null


In [0]:
CREATE OR REPLACE TABLE lab7_lab_kafka_events_flattened
AS
SELECT
  key,
  timestamp,
  parsed:user_id::STRING AS user_id,
  parsed:event_type::STRING AS event_type,
  parsed:event_timestamp::TIMESTAMP AS event_timestamp,
  parsed:items AS items
FROM (
  SELECT
    key,
    timestamp,
    parse_json(decoded_value) AS parsed
  FROM lab7_lab_kafka_events_raw
);

SELECT * FROM lab7_lab_kafka_events_flattened

key,timestamp,user_id,event_type,event_timestamp,items
event_0,1.774387532258797E9,user_1824,click,2026-03-24T21:25:32.258732Z,"[{""item_id"":""item_936"",""price_usd"":64.81,""quantity"":1},{""item_id"":""item_243"",""price_usd"":66.66,""quantity"":3},{""item_id"":""item_801"",""price_usd"":68.78,""quantity"":4}]"
event_1,1.774387532258836E9,user_6398,view,2026-03-24T21:25:32.258804Z,"[{""item_id"":""item_824"",""price_usd"":44.04,""quantity"":2},{""item_id"":""item_758"",""price_usd"":78.67,""quantity"":2},{""item_id"":""item_573"",""price_usd"":32.62,""quantity"":1}]"
event_2,1.774387532258861E9,user_4537,click,2026-03-24T21:25:32.25884Z,"[{""item_id"":""item_934"",""price_usd"":99.28,""quantity"":4},{""item_id"":""item_806"",""price_usd"":90.42,""quantity"":3}]"
event_3,1.774387532258889E9,user_7999,view,2026-03-24T21:25:32.258865Z,"[{""item_id"":""item_779"",""price_usd"":24.62,""quantity"":1},{""item_id"":""item_557"",""price_usd"":76.41,""quantity"":4},{""item_id"":""item_202"",""price_usd"":83.78,""quantity"":3}]"
event_4,1.774387532258913E9,user_1808,view,2026-03-24T21:25:32.258893Z,"[{""item_id"":""item_627"",""price_usd"":74.17,""quantity"":4},{""item_id"":""item_467"",""price_usd"":16.92,""quantity"":4}]"
event_5,1.774387532258931E9,user_8012,click,2026-03-24T21:25:32.258916Z,"[{""item_id"":""item_139"",""price_usd"":99.43,""quantity"":3}]"
event_6,1.774387532258961E9,user_1878,purchase,2026-03-24T21:25:32.258935Z,"[{""item_id"":""item_530"",""price_usd"":95.02,""quantity"":3},{""item_id"":""item_185"",""price_usd"":94.89,""quantity"":1},{""item_id"":""item_675"",""price_usd"":44.4,""quantity"":2}]"
event_7,1.774387532258983E9,user_3198,view,2026-03-24T21:25:32.258964Z,"[{""item_id"":""item_212"",""price_usd"":71.04,""quantity"":1},{""item_id"":""item_121"",""price_usd"":11.26,""quantity"":4}]"
event_8,1.774387532259001E9,user_1069,click,2026-03-24T21:25:32.258986Z,"[{""item_id"":""item_477"",""price_usd"":58.04,""quantity"":3}]"
event_9,1.774387532259023E9,user_8884,purchase,2026-03-24T21:25:32.259004Z,"[{""item_id"":""item_403"",""price_usd"":88.23,""quantity"":3},{""item_id"":""item_146"",""price_usd"":82.83,""quantity"":3}]"


In [0]:
DESCRIBE lab7_lab_kafka_events_flattened;

col_name,data_type,comment
key,string,null
timestamp,double,null
user_id,string,null
event_type,string,null
event_timestamp,timestamp,null
items,variant,null


In [0]:
CREATE OR REPLACE TABLE items_unpacked
AS
SELECT
  key,
  to_timestamp(timestamp / 1000) AS timestamp,
  user_id,
  item.item_id AS item_id,
  item.price_usd AS price_usd,
  item.quantity AS quantity
FROM lab7_lab_kafka_events_flattened
LATERAL VIEW EXPLODE(from_json(CAST(items AS STRING), 'ARRAY<STRUCT<item_id: STRING, price_usd: DOUBLE, quantity: INT>>')) AS item;

SELECT * FROM items_unpacked

key,timestamp,user_id,item_id,price_usd,quantity
event_0,1970-01-21T12:53:07.532258Z,user_1824,item_936,64.81,1
event_0,1970-01-21T12:53:07.532258Z,user_1824,item_243,66.66,3
event_0,1970-01-21T12:53:07.532258Z,user_1824,item_801,68.78,4
event_1,1970-01-21T12:53:07.532258Z,user_6398,item_824,44.04,2
event_1,1970-01-21T12:53:07.532258Z,user_6398,item_758,78.67,2
event_1,1970-01-21T12:53:07.532258Z,user_6398,item_573,32.62,1
event_2,1970-01-21T12:53:07.532258Z,user_4537,item_934,99.28,4
event_2,1970-01-21T12:53:07.532258Z,user_4537,item_806,90.42,3
event_3,1970-01-21T12:53:07.532258Z,user_7999,item_779,24.62,1
event_3,1970-01-21T12:53:07.532258Z,user_7999,item_557,76.41,4


In [0]:
DESCRIBE items_unpacked;

col_name,data_type,comment
key,string,null
timestamp,timestamp,null
user_id,string,null
item_id,string,null
price_usd,double,null
quantity,int,null


In [0]:
CREATE OR REPLACE TABLE items_unpacked_silver
AS
SELECT
  key,
  date_format(
    to_timestamp(timestamp / 1000),
    'yyyy-MM-dd HH:mm:ss'
  ) AS timestamp_formatted,
  regexp_extract(user_id, '\\d+', 0) AS user_id_trimmed,
  regexp_extract(item.item_id, '\\d+', 0) AS item_id_trimmed,
  item.price_usd AS price_usd,
  item.quantity AS quantity
FROM lab7_lab_kafka_events_flattened
LATERAL VIEW EXPLODE(from_json(CAST(items AS STRING), 'ARRAY<STRUCT<item_id: STRING, price_usd: DOUBLE, quantity: INT>>')) AS item;

SELECT * FROM items_unpacked_silver

key,timestamp_formatted,user_id_trimmed,item_id_trimmed,price_usd,quantity
event_0,1970-01-21 12:53:07,1824,936,64.81,1
event_0,1970-01-21 12:53:07,1824,243,66.66,3
event_0,1970-01-21 12:53:07,1824,801,68.78,4
event_1,1970-01-21 12:53:07,6398,824,44.04,2
event_1,1970-01-21 12:53:07,6398,758,78.67,2
event_1,1970-01-21 12:53:07,6398,573,32.62,1
event_2,1970-01-21 12:53:07,4537,934,99.28,4
event_2,1970-01-21 12:53:07,4537,806,90.42,3
event_3,1970-01-21 12:53:07,7999,779,24.62,1
event_3,1970-01-21 12:53:07,7999,557,76.41,4


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>